### Install libraries

In [1]:
!pip install transformers datasets scikit-learn


### Import libraries

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch.nn as nn
from tqdm import tqdm


### Device

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


### Load Dataset

In [4]:
dataset = load_dataset("go_emotions", "simplified")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


### Verify Number of labels

In [5]:
num_labels = len(dataset["train"].features["labels"].feature.names)
print("Number of labels:", num_labels)


Number of labels: 28


###  Multi-Hot Encoding

In [6]:
def encode_labels(example):
    multi_hot = [0.0] * num_labels
    for label_idx in example["labels"]:
        multi_hot[label_idx] = 1.0
    return {"labels": multi_hot}

encoded_train = dataset["train"].map(
    encode_labels,
    remove_columns=["labels"]
)

encoded_val = dataset["validation"].map(
    encode_labels,
    remove_columns=["labels"]
)

### Load RoBERTa Tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


### Tokenization Function

In [8]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )


### Tokenize Train & Validation

In [9]:
tokenized_train = encoded_train.map(tokenize_function, batched=True)
tokenized_val = encoded_val.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text", "id"])
tokenized_val = tokenized_val.remove_columns(["text", "id"])


Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

### Force Float32 Labels

In [10]:
from datasets import Features, Sequence, Value

new_features = Features({
    "input_ids": Sequence(Value("int32")),
    "attention_mask": Sequence(Value("int8")),
    "labels": Sequence(Value("float32")),
})

tokenized_train = tokenized_train.cast(new_features)
tokenized_val = tokenized_val.cast(new_features)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")


Casting the dataset:   0%|          | 0/5426 [00:00<?, ? examples/s]

### Create DataLoaders

In [11]:
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_dataloader = DataLoader(
    tokenized_val,
    batch_size=16,
    num_workers=2,
    pin_memory=True
)

print("Dataloaders ready")


Dataloaders ready


### Load RoBERTa Model

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=num_labels
)

model.to(device)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

### Loss Function

In [13]:
criterion = nn.BCEWithLogitsLoss()


### Optimizer

In [14]:
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 2e-05
    maximize: False
    weight_decay: 0.01
)

### Training Loop

In [15]:
epochs = 2

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # ----- TRAIN -----
    model.train()
    total_train_loss = 0

    train_progress = tqdm(train_dataloader)

    for batch in train_progress:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        loss = criterion(logits, labels)

        total_train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_progress.set_postfix(loss=loss.item())

    avg_train_loss = total_train_loss / len(train_dataloader)

    # ----- VALIDATION -----
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            logits = outputs.logits
            loss = criterion(logits, labels)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_dataloader)

    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")



Epoch 1/2


100%|██████████| 2714/2714 [16:01<00:00,  2.82it/s, loss=0.0478]


Training Loss: 0.1198
Validation Loss: 0.0912

Epoch 2/2


100%|██████████| 2714/2714 [16:02<00:00,  2.82it/s, loss=0.0274]


Training Loss: 0.0869
Validation Loss: 0.0850


### Evauation Metrics

In [16]:
from sklearn.metrics import f1_score

model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

all_probs = torch.cat(all_probs).numpy()
all_labels = torch.cat(all_labels).numpy()

# Default threshold 0.5
preds = (all_probs > 0.5).astype(int)

micro_f1 = f1_score(all_labels, preds, average="micro")
macro_f1 = f1_score(all_labels, preds, average="macro")

print("Micro F1:", micro_f1)
print("Macro F1:", macro_f1)


Micro F1: 0.5630123927550048
Macro F1: 0.38865967827884923


### Threshold Tuning

In [17]:
from sklearn.metrics import f1_score
import numpy as np

best_threshold = 0
best_micro = 0

for threshold in np.arange(0.1, 0.9, 0.05):
    preds = (all_probs > threshold).astype(int)
    micro = f1_score(all_labels, preds, average="micro")

    if micro > best_micro:
        best_micro = micro
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best Micro F1:", best_micro)


Best Threshold: 0.30000000000000004
Best Micro F1: 0.6056469128141483


In [18]:
preds = (all_probs > best_threshold).astype(int)

micro_f1 = f1_score(all_labels, preds, average="micro")
macro_f1 = f1_score(all_labels, preds, average="macro")

print("Optimized Micro F1:", micro_f1)
print("Optimized Macro F1:", macro_f1)


Optimized Micro F1: 0.6056469128141483
Optimized Macro F1: 0.45948661814543273


In [19]:
model.save_pretrained("emotion_model_roberta_best")
tokenizer.save_pretrained("emotion_model_roberta_best")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_model_roberta_best/tokenizer_config.json',
 'emotion_model_roberta_best/tokenizer.json')

In [20]:
!zip -r emotion_model_roberta_best.zip emotion_model_roberta_best


  adding: emotion_model_roberta_best/ (stored 0%)
  adding: emotion_model_roberta_best/tokenizer_config.json (deflated 50%)
  adding: emotion_model_roberta_best/config.json (deflated 65%)
  adding: emotion_model_roberta_best/model.safetensors (deflated 11%)
  adding: emotion_model_roberta_best/tokenizer.json (deflated 82%)


In [21]:
from google.colab import files
files.download("emotion_model_roberta_best.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
BEST_THRESHOLD = 0.30
print("Using threshold:", BEST_THRESHOLD)


Using threshold: 0.3
